In [1]:
# %%
import json
import re
import pathlib
import multiprocessing as mp
import xarray as xr
import tqdm

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from functools import partial
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

In [2]:
# %%
src_path = pathlib.Path('/gpfs/work2/0/prjs1027/swn-runs/')
boundary_path = pathlib.Path('/gpfs/work2/0/prjs1027/guus/data/BoundaryConditions/bnd_storms.nc')
write_dir = pathlib.Path('/gpfs/work2/0/prjs1027/tfrecords/run2')
write_dir.mkdir(exist_ok=True, parents=True)
filtered_paths = [path for path in src_path.glob('ens*') if re.search(r'\d+$', str(path))]
filtered_paths = [path for path in src_path.glob('0*') if path.stem[-4:] != '2022']

In [3]:
# %%
boundary_ds = xr.open_dataset(boundary_path)
dummy_ds = xr.open_dataset(next(filtered_paths[0].glob('computations/ens0/*/output/*.nc')))

In [4]:
dummy_ds

<xarray.Dataset>
Dimensions:    (time: 25, longitude: 421, latitude: 481)
Coordinates:
  * time       (time) datetime64[ns] 2023-02-17 ... 2023-02-20
  * longitude  (longitude) float32 -12.0 -11.95 -11.9 -11.85 ... 8.9 8.95 9.0
  * latitude   (latitude) float32 48.0 48.03 48.07 48.1 ... 63.93 63.97 64.0
Data variables:
    hs         (time, latitude, longitude) float32 ...
    hswe       (time, latitude, longitude) float32 ...
    tmm10      (time, latitude, longitude) float32 ...
    tps        (time, latitude, longitude) float32 ...
    theta0     (time, latitude, longitude) float32 ...
    fdir       (time, latitude, longitude) float32 ...
    spread     (time, latitude, longitude) float32 ...
    ssh        (time, latitude, longitude) float32 ...
    xcur       (time, latitude, longitude) float32 ...
    ycur       (time, latitude, longitude) float32 ...
    xwnd       (time, latitude, longitude) float32 ...
    ywnd       (time, latitude, longitude) float32 ...
Attributes:
    Conventions:             CF-1.5
    History:                 Created with agioncmd version 1.5
    Directional_convention:  nautical
    project:                 SWAN-EPS
    run:                     000

In [5]:
# %%
output_variables = ["hs", "theta0_x", "theta0_y", "tmm10", "tps", "hswe"]
input_variables = ["xwnd", "ywnd", "ssh", "xcur", "ycur"]
boundary_variables = ["hs", "hswe", "tmm10"]
conversion_dict = {"hs": "Hm0", "hswe": "HE10", "tmm10": "Tmm10"}

In [7]:
# %%
def compute_min_max(path):
    all_datasets = []
    min_values = {}
    max_values = {}
    
    ens_paths_50 = [ens_path for ens_path in path.glob('computations/ens*') if re.search(r'\d+$', str(ens_path))]
    ens_paths = []
    for path in ens_paths_50:
        ens = re.search(r'\d+$', str(path)).group(0)
        if int(ens) in range(25):
            # ens_paths.append(path)
            
            for var in dummy_ds.variables:
                min_values[var] = []
                max_values[var] = []
            
            for _ in tqdm.tqdm(list(path.glob('*'))):
                try:
                    ds_path = next(_.glob('output/*.nc'))
                    ds = xr.open_dataset(ds_path)
                    for var in ds.variables:
                        min_values[var].append(ds[var].min().values)
                        max_values[var].append(ds[var].max().values)
                except:
                    continue
        
            min_max_values = {}
            for var in dummy_ds.variables:
                min_value = np.nanmin(min_values[var])
                max_value = np.nanmax(max_values[var])
                min_max_values[var] = np.array([min_value, max_value])
            return min_max_values

In [8]:
# %%
min_max_values = {}
for paths in filtered_paths:
    min_max_values[paths.stem] = compute_min_max(paths)

100%|██████████| 30/30 [00:15<00:00,  1.88it/s]
/scratch-local/gvhemert1.8104105/ipykernel_112502/1572366413.py:30: RuntimeWarning: All-NaN axis encountered
  min_value = np.nanmin(min_values[var])
/scratch-local/gvhemert1.8104105/ipykernel_112502/1572366413.py:31: RuntimeWarning: All-NaN axis encountered
  max_value = np.nanmax(max_values[var])
100%|██████████| 6/6 [00:02<00:00,  2.04it/s]


In [9]:
# %%
min_max_values_detailed = {}
for ens in min_max_values.keys():
    for var in min_max_values[ens].keys():
        try:
            min_max_values_detailed[var] = np.concatenate([min_max_values_detailed[var], min_max_values[ens][var]])
        except:
            min_max_values_detailed[var] = min_max_values[ens][var]

In [10]:
# %%
for var in min_max_values_detailed.keys():
    min_value = np.nanmin(min_max_values_detailed[var])
    max_value = np.nanmax(min_max_values_detailed[var])
    min_max_values_detailed[var] = {
        "minimum": min_value,
        "maximum": max_value
    }

In [11]:
min_max_values_detailed

{'time': {'minimum': numpy.datetime64('2013-12-01T00:00:00.000000000'),
  'maximum': numpy.datetime64('2023-12-31T00:00:00.000000000')},
 'longitude': {'minimum': -12.0, 'maximum': 9.0},
 'latitude': {'minimum': 48.0, 'maximum': 64.0},
 'hs': {'minimum': 0.0, 'maximum': 13.415173530578613},
 'hswe': {'minimum': 0.0, 'maximum': 12.44392204284668},
 'tmm10': {'minimum': 0.0, 'maximum': 50.0},
 'tps': {'minimum': 1.6388435363769531, 'maximum': 33.33308029174805},
 'theta0': {'minimum': 0.0, 'maximum': 360.0},
 'fdir': {'minimum': 0.0, 'maximum': 360.0},
 'spread': {'minimum': 0.0, 'maximum': 81.02664184570312},
 'ssh': {'minimum': -6.4157538414001465, 'maximum': 7.284157752990723},
 'xcur': {'minimum': -18.756065368652344, 'maximum': 15.502792358398438},
 'ycur': {'minimum': -18.756065368652344, 'maximum': 14.77645206451416},
 'xwnd': {'minimum': -27.176733016967773, 'maximum': 32.59987258911133},
 'ywnd': {'minimum': -33.817562103271484, 'maximum': 29.75249481201172}}

In [12]:
# %%
df = pd.DataFrame(min_max_values_detailed)

In [13]:
# %%
ds_min_max = xr.Dataset.from_dataframe(df)
ds_min_max_dir = write_dir / "min-max-values"
ds_min_max_dir.mkdir(exist_ok=True, parents=True)
ds_min_max.to_netcdf(ds_min_max_dir / "min-max-values-storms.nc")

In [14]:
ds_min_max

<xarray.Dataset>
Dimensions:    (index: 2)
Coordinates:
  * index      (index) object 'minimum' 'maximum'
Data variables: (12/15)
    time       (index) datetime64[ns] 2013-12-01 2023-12-31
    longitude  (index) float64 -12.0 9.0
    latitude   (index) float64 48.0 64.0
    hs         (index) float64 0.0 13.42
    hswe       (index) float64 0.0 12.44
    tmm10      (index) float64 0.0 50.0
    ...         ...
    spread     (index) float64 0.0 81.03
    ssh        (index) float64 -6.416 7.284
    xcur       (index) float64 -18.76 15.5
    ycur       (index) float64 -18.76 14.78
    xwnd       (index) float64 -27.18 32.6
    ywnd       (index) float64 -33.82 29.75

In [15]:
ds_min_max_prev = xr.open_dataset(ds_min_max_dir / 'min-max-values.nc')

In [16]:
ds_min_max_all = ds_min_max.copy(deep=True)
for var in ds_min_max.variables:
    ds_min_max_all[var][0] = min(ds_min_max[var][0], ds_min_max_prev[var][0])
    ds_min_max_all[var][1] = max(ds_min_max[var][1], ds_min_max_prev[var][1])

TypeError: IndexVariable values cannot be modified

In [22]:
min_max_values_detailed = {}
for var in ds_min_max.variables:
    min_value = min(ds_min_max[var].values[0], ds_min_max_prev[var].values[0])
    max_value = max(ds_min_max[var].values[1], ds_min_max_prev[var].values[1])
    min_max_values_detailed[var] = {
        "minimum": min_value,
        "maximum": max_value
    }

In [23]:
df_all = pd.DataFrame(min_max_values_detailed)
ds_min_max_all = xr.Dataset.from_dataframe(df_all)
ds_min_max.to_netcdf(ds_min_max_dir / "min-max-values-all.nc")

In [24]:
ds_min_max_all

<xarray.Dataset>
Dimensions:    (index: 2)
Coordinates:
  * index      (index) object 'minimum' 'maximum'
Data variables: (12/15)
    time       (index) datetime64[ns] 2013-12-01 2023-12-31
    longitude  (index) float64 -12.0 9.0
    latitude   (index) float64 48.0 64.0
    hs         (index) float64 0.0 15.95
    hswe       (index) float64 0.0 14.69
    tmm10      (index) float64 0.0 50.0
    ...         ...
    spread     (index) float64 0.0 81.03
    ssh        (index) float64 -6.416 7.284
    xcur       (index) float64 -18.76 15.5
    ycur       (index) float64 -18.76 14.78
    xwnd       (index) float64 -27.42 36.29
    ywnd       (index) float64 -34.12 46.39

In [26]:
ds_min_max_prev['ywnd'].values

array([-34.11969376,  46.38508224])

In [25]:
ds_min_max

<xarray.Dataset>
Dimensions:    (index: 2)
Coordinates:
  * index      (index) object 'minimum' 'maximum'
Data variables: (12/15)
    time       (index) datetime64[ns] 2013-12-01 2023-12-31
    longitude  (index) float64 -12.0 9.0
    latitude   (index) float64 48.0 64.0
    hs         (index) float64 0.0 13.42
    hswe       (index) float64 0.0 12.44
    tmm10      (index) float64 0.0 50.0
    ...         ...
    spread     (index) float64 0.0 81.03
    ssh        (index) float64 -6.416 7.284
    xcur       (index) float64 -18.76 15.5
    ycur       (index) float64 -18.76 14.78
    xwnd       (index) float64 -27.18 32.6
    ywnd       (index) float64 -33.82 29.75

In [41]:
min_max_values_detailed = {}
for ens in min_max_values.keys():
    for var in min_max_values[ens].keys():
        try:
            min_max_values_detailed[var] = np.concatenate([min_max_values_detailed[var], min_max_values[ens][var]])
        except Exception as e:
            print(e)
            min_max_values_detailed[var] = min_max_values[ens][var]

'time'
'longitude'
'latitude'
'hs'
'hswe'
'tmm10'
'tps'
'theta0'
'fdir'
'spread'
'ssh'
'xcur'
'ycur'
'xwnd'
'ywnd'


In [48]:
min_max_values_detailed

{'time': {'minimum': numpy.datetime64('2013-12-01T00:00:00.000000000'),
  'maximum': numpy.datetime64('2013-12-07T00:00:00.000000000')},
 'longitude': {'minimum': -12.0, 'maximum': 9.0},
 'latitude': {'minimum': 48.0, 'maximum': 64.0},
 'hs': {'minimum': 0.0, 'maximum': 11.11331558227539},
 'hswe': {'minimum': 0.0, 'maximum': 9.90783405303955},
 'tmm10': {'minimum': 0.00152587890625, 'maximum': 49.959564208984375},
 'tps': {'minimum': 1.6388435363769531, 'maximum': 33.33308029174805},
 'theta0': {'minimum': 0.0, 'maximum': 360.0},
 'fdir': {'minimum': 0.0, 'maximum': 360.0},
 'spread': {'minimum': 0.0, 'maximum': 81.02664184570312},
 'ssh': {'minimum': -5.518051624298096, 'maximum': 5.944700241088867},
 'xcur': {'minimum': -18.756065368652344, 'maximum': 14.77523136138916},
 'ycur': {'minimum': -18.756065368652344, 'maximum': 14.77523136138916},
 'xwnd': {'minimum': -16.470840454101562, 'maximum': 32.038330078125},
 'ywnd': {'minimum': -33.817562103271484, 'maximum': 20.508438110351562